# 3) Machine learning — Exercises

**Goals:** audit features for availability, find a subtle leak in a rolling feature, compare validation schemes against the truth, see whether they pick different models, and choose an ensemble weight honestly.

The setup cell holds the lecture's honest feature table `FEATURES`, the issue time of every hour (`ISSUE`, 11:00 on D−1), a `boosted()` model factory and `mae`. Demand in MW.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
demand, idx = df["demand"], df.index
dow = idx.dayofweek
daytype = np.where(df.holiday | (dow == 6), 2, np.where(dow == 5, 1, 0))
ISSUE = idx.normalize() - pd.Timedelta(hours=13)                 # 11:00 on D-1: the last known hour
FEATURES = pd.DataFrame({
    "hour": idx.hour, "daytype": daytype, "day_of_year": idx.dayofyear,
    "temperature_forecast": df.temperature_forecast,
    "last_known": demand.reindex(ISSUE).to_numpy(),
    "same_hour_2_days_ago": demand.shift(48), "same_hour_last_week": demand.shift(168)}, index=idx)


def boosted(**kw):
    """The gradient-boosting model used throughout (slightly faster settings than the lecture)."""
    return HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, random_state=0, **kw)


def mae(forecast, actual):
    return float(np.abs(np.asarray(forecast) - np.asarray(actual)).mean())

### Warm-ups

**1. A feature audit**

Write `audit(lags, issue_hour=12)`: for a dict {feature name: lag in hours}, return the share of the 24 target hours of a day-ahead forecast for which each lag is available (the lecture's rule: lag ≥ target hour + 25 − issue hour). Audit 1, 24, 48 and 168 hours, then the same lags for a forecast issued at 08:00.

In [2]:
def audit(lags: dict, issue_hour: int = 12) -> pd.Series:
    """Share of target hours (0-23 of day D) for which each lag is known at issue_hour on D-1."""
    ### BEGIN SOLUTION
    return pd.Series({name: np.mean([lag >= h + 25 - issue_hour for h in range(24)]) for name, lag in lags.items()})
    ### END SOLUTION

In [3]:
LAGS = {"one_hour_ago": 1, "same_hour_yesterday": 24, "same_hour_2_days_ago": 48, "same_hour_last_week": 168}
noon, early = audit(LAGS), audit(LAGS, issue_hour=8)
assert noon.to_dict() == {"one_hour_ago": 0.0, "same_hour_yesterday": 0.5, "same_hour_2_days_ago": 1.0, "same_hour_last_week": 1.0}
assert early["same_hour_yesterday"] < noon["same_hour_yesterday"], "an earlier issue time leaves less of yesterday"
pd.DataFrame({"issued 12:00": noon, "issued 08:00": early})

                      issued 12:00  issued 08:00
one_hour_ago                   0.0      0.000000
same_hour_yesterday            0.5      0.333333
same_hour_2_days_ago           1.0      1.000000
same_hour_last_week            1.0      1.000000

**2. A leak in a rolling feature**

"Average demand over the last few hours" sounds like harmless history. Build it three ways: the 24-hour and the 3-hour average **ending at the target hour** (`demand.rolling(n).mean()`), and the 24-hour average **ending at 11:00 on D−1**, the last known hour. Add each to `FEATURES`, train `boosted()` on 2023–2024 and score 2025; also score the model without any rolling feature. Write `rolling_feature_test()` returning the four MAEs.

In [4]:
def rolling_feature_test() -> dict:
    """2025 MAE without a rolling feature, with two leaky ones and with an honest one."""
    ### BEGIN SOLUTION
    rolling24 = demand.rolling(24).mean()
    variants = {"no rolling feature": None,
                "leaky: 24 h ending at the target hour": rolling24,
                "leaky: 3 h ending at the target hour": demand.rolling(3).mean(),
                "honest: 24 h ending at 11:00 on D-1": pd.Series(rolling24.reindex(ISSUE).to_numpy(), index=idx)}
    out = {}
    for name, feature in variants.items():
        X = (FEATURES if feature is None else FEATURES.assign(rolling_mean=feature)).dropna()
        y = demand.loc[X.index]
        tr = X.index < "2025"
        out[name] = mae(boosted().fit(X[tr], y[tr]).predict(X[~tr]), y[~tr])
    return out
    ### END SOLUTION

In [5]:
rolling_result = pd.Series(rolling_feature_test())
base = rolling_result["no rolling feature"]
assert rolling_result["leaky: 24 h ending at the target hour"] < 0.9 * base, "the leak flatters"
shorter, longer = rolling_result["leaky: 3 h ending at the target hour"], rolling_result["leaky: 24 h ending at the target hour"]
assert shorter < 0.5 * longer, "the shorter the window, the louder the leak"
assert abs(rolling_result["honest: 24 h ending at 11:00 on D-1"] - base) < 0.05 * base, "the honest version adds little"
rolling_result.round(1)

no rolling feature                       76.9
leaky: 24 h ending at the target hour    62.0
leaky: 3 h ending at the target hour     21.4
honest: 24 h ending at 11:00 on D-1      77.5
dtype: float64

### Core

**3. Which validation scheme tells the truth?**

Using only 2023–2024, estimate the error the model will make on 2025 with three schemes: (a) shuffled 5-fold cross-validation, (b) a **hold-out year** — train on 2023, validate on 2024, (c) `TimeSeriesSplit(n_splits=4, test_size=2190)`, whose first fold still trains on about a year. Then train on 2023–2024 and measure the real 2025 error. Write `validation_schemes(**model_kw)` returning the three estimates and the truth.

In [6]:
def validation_schemes(**model_kw) -> dict:
    """Estimated (from 2023-2024 only) and actual 2025 MAE of boosted(**model_kw)."""
    ### BEGIN SOLUTION
    X = FEATURES.dropna()
    y = demand.loc[X.index]
    tr = X.index < "2025"
    Xtr, ytr = X[tr], y[tr]
    out = {}
    for name, cv in {"shuffled 5-fold": KFold(5, shuffle=True, random_state=0),
                     "TimeSeriesSplit, >= 1 year of training": TimeSeriesSplit(n_splits=4, test_size=2190)}.items():
        out[name] = np.mean([mae(boosted(**model_kw).fit(Xtr.iloc[a], ytr.iloc[a]).predict(Xtr.iloc[b]), ytr.iloc[b])
                             for a, b in cv.split(Xtr)])
    fit, valid = Xtr.index < "2024", Xtr.index >= "2024"
    out["hold-out year (2023 -> 2024)"] = mae(boosted(**model_kw).fit(Xtr[fit], ytr[fit]).predict(Xtr[valid]), ytr[valid])
    out["actual 2025"] = mae(boosted(**model_kw).fit(Xtr, ytr).predict(X[~tr]), y[~tr])
    return out
    ### END SOLUTION

In [7]:
schemes = pd.Series(validation_schemes())
truth = schemes["actual 2025"]
error = (schemes.drop("actual 2025") - truth).abs()
assert error.idxmax() == "shuffled 5-fold", "shuffled folds are the furthest from the truth"
pd.DataFrame({"MAE": schemes, "off by": (schemes - truth)}).round(1)

                                         MAE  off by
shuffled 5-fold                         46.7   -30.2
TimeSeriesSplit, >= 1 year of training  78.1     1.1
hold-out year (2023 -> 2024)            78.7     1.8
actual 2025                             76.9     0.0

**4. Do the schemes pick different models?**

For `max_leaf_nodes` in [7, 31, 127] (small to large trees), run `validation_schemes`. Which tree size would each scheme choose, and what is the 2025 error of each choice?

In [8]:
def model_choice(sizes=(7, 31, 127)) -> pd.DataFrame:
    """Estimated and actual MAE for each tree size."""
    ### BEGIN SOLUTION
    return pd.DataFrame({size: validation_schemes(max_leaf_nodes=size) for size in sizes}).T.rename_axis("max_leaf_nodes")
    ### END SOLUTION

In [9]:
choices = model_choice()
picked = {scheme: int(choices[scheme].idxmin()) for scheme in choices.columns if scheme != "actual 2025"}
print("tree size chosen by each scheme:", picked, "| best on 2025:", int(choices["actual 2025"].idxmin()))
assert picked["shuffled 5-fold"] == choices["shuffled 5-fold"].idxmin()
choices.round(1)

tree size chosen by each scheme: {'shuffled 5-fold': 127, 'TimeSeriesSplit, >= 1 year of training': 7, 'hold-out year (2023 -> 2024)': 7} | best on 2025: 7


                shuffled 5-fold  TimeSeriesSplit, >= 1 year of training  \
max_leaf_nodes                                                            
7                          61.2                                    73.7   
31                         46.7                                    78.1   
127                        39.9                                    80.2   

                hold-out year (2023 -> 2024)  actual 2025  
max_leaf_nodes                                             
7                                       74.2         75.8  
31                                      78.7         76.9  
127                                     80.8         79.2  

**5. Interpretation**

Why do the rolling features leak, although "the average of the last hours" sounds like history — and why does the 3-hour version leak so much more than the 24-hour one? Which validation scheme came closest to the truth, and did the choice of scheme change the model you would have picked?

<!-- BEGIN SOLUTION -->
A rolling mean "ending at the target hour" is history only from the target hour's point of view. A day-ahead forecast is made at noon on D−1, and every hour after 11:00 on D−1 is still in the future then — so a window ending at the target hour is made almost entirely of values the forecaster cannot have. How loudly the leak shows depends on the window: the 3-hour mean is mostly the target hour and its two neighbours, so the model copies it and reaches a fake 21.4 MW; the 24-hour mean dilutes the target among 23 other hours, but still flatters by about a fifth (62.0 against 76.9 MW). The honest version, ending at 11:00 on D−1, adds nothing (77.5 MW): the recent level it carries is already in `last_known`. A leak does not have to be dramatic to mislead — a 20 % improvement looks like a good feature, not like a bug.

Shuffled cross-validation was off by −30 MW; the hold-out year and TimeSeriesSplit with a year of training were within 2 MW of the truth. Worse, the shuffled scheme **picked the wrong model**: it rewards the largest trees (127 leaves), which memorise near-identical neighbouring hours, while those trees are the worst on 2025 (79.2 MW). Both honest schemes chose the smallest trees (7 leaves), which really are best (75.8 MW). A validation scheme is not only a way to report the error; it is the instrument that chooses the model — and a biased instrument makes biased choices.
<!-- END SOLUTION -->

### Challenge

**6. An ensemble weight, chosen honestly**

The lecture averaged the regression and the boosting model 50/50. Choose the weight from data instead: fit both models on 2023, pick w ∈ {0, 0.1, …, 1} for w · regression + (1 − w) · boosting by MAE on 2024, then refit both on 2023–2024 and score 2025 with that w and with 0.5. Write `ensemble_weight()`.

In [10]:
REG = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
reg_frame = df.assign(daytype=daytype, hour=idx.hour, month=idx.month)


def forecasts(fit_until: str, predict_year: str) -> tuple[pd.Series, pd.Series]:
    """Regression and boosting forecasts for predict_year, both fitted on data up to fit_until."""
    X = FEATURES.dropna()
    y = demand.loc[X.index]
    fit = X.index <= pd.Timestamp(fit_until + "-12-31 23:00")
    target = X.index[X.index.year == int(predict_year)]
    gbm = pd.Series(boosted().fit(X[fit], y[fit]).predict(X.loc[target]), index=target)
    reg_model = smf.ols(REG, reg_frame.loc[:fit_until]).fit()
    reg = reg_model.predict(reg_frame.assign(temperature=reg_frame.temperature_forecast)).loc[target]
    return reg, gbm


def ensemble_weight(weights=np.round(np.linspace(0, 1, 11), 1)) -> dict:
    """Weight on the regression chosen on 2024, and the 2025 MAE with that weight and with 0.5."""
    ### BEGIN SOLUTION
    reg, gbm = forecasts("2023", "2024")
    actual = demand.loc[reg.index]
    valid = pd.Series({w: mae(w * reg + (1 - w) * gbm, actual) for w in weights})
    w = float(valid.idxmin())
    reg, gbm = forecasts("2024", "2025")
    actual = demand.loc[reg.index]
    return {"chosen_w": w, "MAE_2025_chosen": mae(w * reg + (1 - w) * gbm, actual),
            "MAE_2025_half": mae(0.5 * reg + 0.5 * gbm, actual),
            "MAE_2025_regression": mae(reg, actual), "MAE_2025_boosting": mae(gbm, actual)}
    ### END SOLUTION

In [11]:
result = ensemble_weight()
assert 0 < result["chosen_w"] < 1
assert result["MAE_2025_chosen"] < min(result["MAE_2025_regression"], result["MAE_2025_boosting"])
pd.Series(result).round(2)

chosen_w                0.60
MAE_2025_chosen        68.76
MAE_2025_half          68.75
MAE_2025_regression    73.56
MAE_2025_boosting      76.93
dtype: float64

**7. Interpretation**

Did choosing the weight on 2024 beat the simple 50/50? Why is "equal weights" such a hard benchmark to beat in practice?

<!-- BEGIN SOLUTION -->
No. The weight chosen on 2024 (0.6 on the regression) gives 68.76 MW on 2025 — the same as the simple 50/50 average (68.75 MW). Both beat the better single model by about 5 MW. Equal weights are notoriously hard to beat — the "forecast combination puzzle" — for three reasons visible here: the optimal weight is estimated from one validation year and carries noise of its own; the best weight for 2024 is not exactly the best for 2025; and the error curve is flat near its minimum: on both years, every weight from 0.4 to 0.7 is within about 1 MW of the best. Estimating a weight only pays when the models differ strongly in quality or when there is much more validation data; otherwise the simple average is the honest default.
<!-- END SOLUTION -->